# True-LoRA: matutake にコーディング能力を付与するチュートリアル

**Target Model:** [summerMC/matutake](https://huggingface.co/summerMC/matutake) (2B, Qwen2)

**Hardware:** Colab Free Tier (T4 GPU)

## Step 1: セットアップ

In [ ]:
!nvidia-smi

In [ ]:
!pip install git+https://github.com/MARVserver/TrueLora.git -q
!pip install transformers accelerate bitsandbytes -q
print("Done!")

In [ ]:
import gc
from pathlib import Path

import torch
from true_lora.adapter import (
    AdapterBank, AdapterSpec, LoraTensorSpec,
    save_peft_adapter,
)
from true_lora.generator import TrueLoraGenerator
from true_lora.text import HashingTextEncoder
from true_lora.train import train_on_adapter_bank
from true_lora.reporting import write_json_report

DEVICE = "cuda"
assert torch.cuda.is_available(), "GPUが利用できません！Runtime > Change runtime type > GPUを選択してください"

def mem():
    print(f"  GPU: {torch.cuda.memory_allocated()/1e6:.0f}MB alloc / {torch.cuda.memory_reserved()/1e6:.0f}MB reserved")

def clear():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

## Step 2: アダプタバンク構築

In [ ]:
# 軽量パラメータ
TEXT_DIM = 128
HIDDEN_DIM = 256
LORA_RANK = 4
LORA_ALPHA = 8.0
MAX_NORM = 4.0

# モデル設定を取得
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained("summerMC/matutake", trust_remote_code=True)
HIDDEN_SIZE = cfg.hidden_size
NUM_ATTENTION_HEADS = cfg.num_attention_heads
NUM_KEY_VALUE_HEADS = getattr(cfg, "num_key_value_heads", NUM_ATTENTION_HEADS)
HEAD_DIM = getattr(cfg, "head_dim", HIDDEN_SIZE // NUM_ATTENTION_HEADS)

# GQAモデルではq_projとv_proj/k_projの出力次元が異なる（KVヘッド数に依存するため）
PROJ_OUT_FEATURES = {
    "q_proj": NUM_ATTENTION_HEADS * HEAD_DIM,
    "k_proj": NUM_KEY_VALUE_HEADS * HEAD_DIM,
    "v_proj": NUM_KEY_VALUE_HEADS * HEAD_DIM,
    "o_proj": HIDDEN_SIZE,
}

# attention のみ、4層
LORA_TARGETS = ["q_proj", "v_proj"]
TARGET_LAYERS = [0, 6, 12, 18]

out_features_used = {t: PROJ_OUT_FEATURES[t] for t in LORA_TARGETS}
print(f"Hidden: {HIDDEN_SIZE}, Layers: {TARGET_LAYERS}, Targets: {LORA_TARGETS}")
print(f"Out features: {out_features_used}")

In [ ]:
lora_specs = []
for li in TARGET_LAYERS:
    for t in LORA_TARGETS:
        name = f"model.layers.{li}.self_attn.{t}"
        lora_specs.append(LoraTensorSpec(name, PROJ_OUT_FEATURES[t], HIDDEN_SIZE, LORA_RANK, LORA_ALPHA))

encoder = HashingTextEncoder(dim=TEXT_DIM)

adapters_config = [
    ("python code generation", 0.15, 0.08, 0.85),
    ("python function implementation", 0.18, 0.09, 0.82),
    ("algorithm implementation", 0.20, 0.10, 0.88),
    ("data structure implementation", 0.17, 0.08, 0.83),
    ("dynamic programming solutions", 0.22, 0.11, 0.86),
    ("code debugging and error fixing", 0.13, 0.06, 0.79),
    ("unit test writing", 0.15, 0.07, 0.77),
    ("performance optimization", 0.18, 0.09, 0.80),
    ("numpy array operations", 0.16, 0.08, 0.82),
    ("torch deep learning code", 0.19, 0.09, 0.85),
    ("design pattern implementation", 0.17, 0.08, 0.78),
    ("object oriented programming", 0.15, 0.07, 0.76),
    ("type hints and annotations", 0.11, 0.05, 0.72),
    ("file io and pathlib usage", 0.10, 0.05, 0.71),
    ("code refactoring and cleanup", 0.12, 0.06, 0.74),
]

adapters = []
for desc, sa, sb, score in adapters_config:
    tensors = {}
    for spec in lora_specs:
        tensors[f"{spec.name}.lora_A.weight"] = torch.full(spec.a_shape, sa)
        tensors[f"{spec.name}.lora_B.weight"] = torch.full(spec.b_shape, sb)
    adapters.append(AdapterSpec(
        description=desc,
        embedding=encoder.encode(desc),
        tensors=tensors,
        metrics={"score": score},
    ))

bank = AdapterBank(adapters)
print(f"Bank: {len(bank.adapters)} adapters, {len(lora_specs)} specs")

## Step 3: トレーニング（GPU上で実行）

In [ ]:
# 生成器を作成 → hyperネットワークのみGPUに移動（TrueLoraGeneratorはnn.Moduleではないため）
generator = TrueLoraGenerator(
    tensor_specs=lora_specs,
    adapter_bank=bank,
    text_dim=TEXT_DIM,
    hidden_dim=HIDDEN_DIM,
    max_tensor_norm=MAX_NORM,
)
generator.hyper = generator.hyper.to(DEVICE)

print(f"Generator on: {next(generator.hyper.parameters()).device}")
print(f"Params: {sum(p.numel() for p in generator.hyper.parameters()):,}")
mem()

In [ ]:
# トレーニング（GPU上で直接実行）
print("Training on GPU...")

optimizer = torch.optim.AdamW(generator.hyper.parameters(), lr=1e-3)

# アダプタのターゲットをGPUに事前転送
gpu_targets = []
gpu_embeddings = []
for a in bank.adapters:
    gpu_targets.append({k: v.float().to(DEVICE) for k, v in a.tensors.items()})
    gpu_embeddings.append(a.embedding.to(DEVICE))

losses = []
for step in range(200):
    idx = step % len(bank.adapters)
    predicted, _ = generator.hyper(gpu_embeddings[idx])
    target = gpu_targets[idx]

    loss = torch.tensor(0.0, device=DEVICE, requires_grad=True)
    for name, t in target.items():
        loss = loss + torch.nn.functional.mse_loss(predicted[name], t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if (step + 1) % 50 == 0:
        print(f"  Step {step+1}: loss={loss.item():.6f}")

print(f"Done! loss={losses[-1]:.6f}")
mem()

## Step 4: LoRAアダプタの生成

In [ ]:
# 生成（アダプタバンクはCPU上のテンソルを保持しているため、hyperをCPUに戻してから実行）
generator.hyper = generator.hyper.to("cpu")

prompt = "python code generation"
state_dict, report = generator.generate(prompt, retrieval_k=8)

print(f"Uncertainty: {report['uncertainty']:.4f}")
print(f"Max retrieval: {report['max_retrieval_score']:.4f}")
print(f"Retrieved:")
for p in report["retrieved_adapters"]:
    print(f"  [{p['rank']}] {p['description']} (w={p['weight']:.3f})")

# 保存（CPUに移動してから）
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)
cpu_state = {k: v.cpu() for k, v in state_dict.items()}
save_peft_adapter(OUTPUT_DIR / "coding_lora.pt", cpu_state, report)
write_json_report(OUTPUT_DIR / "generation_report.json", report)
print(f"Saved: {OUTPUT_DIR / 'coding_lora.pt'}")

## Step 5: マージ

GPUメモリを確保するため、生成器を削除してからマージする。

In [ ]:
# 生成器を削除してメモリ解放
del generator, bank, adapters, gpu_targets, gpu_embeddings, state_dict, losses
clear()
print("Memory cleared")
mem()

In [ ]:
from true_lora.merge import merge_adapter_into_hf_model

merged_dir = OUTPUT_DIR / "matutake_coding_merged"
print("Merging LoRA into matutake (4bit quantization for memory)...")

merge_report = merge_adapter_into_hf_model(
    adapter_path=OUTPUT_DIR / "coding_lora.pt",
    model_name_or_path="summerMC/matutake",
    output_dir=merged_dir,
    dtype="bfloat16",
    device="cpu",
    allow_download=True,
)

print(f"Done! Output: {merge_report['output_dir']}")
print(f"Applied: {merge_report['applied_layers']}")

## Step 6: テスト

In [ ]:
# マージしたモデルをCPUでロード（メモリ節約）
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("Loading merged model (4bit)...")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(str(merged_dir), trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    str(merged_dir),
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)
print(f"Loaded! {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B params")
mem()

In [ ]:
def test_coding(prompt, max_new_tokens=512):
    messages = [
        {"role": "system", "content": "You are a helpful coding assistant."},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.9)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

In [ ]:
print("Test 1: Binary Search")
print("=" * 50)
print(test_coding("Write a Python function that implements binary search. Include type hints."))

In [ ]:
print("Test 2: Dynamic Programming")
print("=" * 50)
print(test_coding("Implement longest common subsequence using dynamic programming."))

In [ ]:
print("Test 3: Japanese")
print("=" * 50)
print(test_coding("Pythonでリストの重複を除いた要素を返す関数を書いて。メソッドを使わずに。"))

## HuggingFace Hubにアップロード（オプション）

In [ ]:
USE_HF_UPLOAD = False
HF_REPO = "summerMC/matutake-coding-lora"

if USE_HF_UPLOAD:
    from huggingface_hub import login, HfApi
    login()
    HfApi().upload_folder(folder_path=str(merged_dir), repo_id=HF_REPO, repo_type="model")
    print(f"Uploaded: https://huggingface.co/{HF_REPO}")
else:
    print(f"Manual: huggingface-cli upload {HF_REPO} {merged_dir} .")

## まとめ

| 項目 | 値 |
| --- | --- |
| LoRA Rank | 4 |
| LoRA Alpha | 8.0 |
| Target Layers | 4 (0, 6, 12, 18) |
| LoRA Targets | q_proj, v_proj |
| LoRA Specs | 8 |
| Adapter Bank | 15 |
| Training | 200 steps (GPU) |
| Model Load | 4bit quantization |